# 🚽 トイレきれい度マップ - スクレイパー

Google Colabで全国のトイレデータを収集 → GitHubにpush → Streamlit Cloudに自動反映

**使い方:**
1. 事前にGitHub Personal Access Tokenを作成（repo権限）
2. 各セルを上から順に実行
3. 最後にpushでStreamlit Cloudに反映

In [ ]:
# ============================================================
# Cell 1: 初期セットアップ（初回のみ）
# ============================================================

# Google Driveをマウント（進捗・結果の永続化）
from google.colab import drive
drive.mount('/content/drive')

# ベースディレクトリ
import os
BASE = '/content/drive/MyDrive/toilet-map'
os.makedirs(f'{BASE}/data', exist_ok=True)
os.makedirs(f'{BASE}/raw_parts', exist_ok=True)
os.makedirs(f'{BASE}/progress', exist_ok=True)
os.makedirs(f'{BASE}/queries', exist_ok=True)

print('セットアップ完了')
print(f'ベースディレクトリ: {BASE}')

In [ ]:
# ============================================================
# Cell 2: リポジトリクローン（初回のみ）
# ============================================================

%cd /content

import os
if not os.path.exists('/content/toilet-map/.git'):
    !git clone https://github.com/kaenozu/toilet-map.git
else:
    %cd /content/toilet-map
    !git pull

%cd /content/toilet-map
print('リポジトリ準備完了')

In [ ]:
# ============================================================
# Cell 3: スクレイパーインストール（初回のみ）
# ============================================================

# google-maps-scraperをインストール
!pip install google-maps-scraper 2>/dev/null || echo 'pip install failed, trying go...'

# Goが必要（scraperがGo製）
!apt-get install -y golang-go > /dev/null 2>&1

# 直接ビルド
import os, subprocess
scraper_dir = '/content/gmaps-scraper'
if not os.path.exists(scraper_dir):
    !cd /content && git clone https://github.com/gosom/google-maps-scraper.git gmaps-scraper
    !cd /content/gmaps-scraper && go build -o /usr/local/bin/google-maps-scraper

# 動作確認
result = subprocess.run(['google-maps-scraper', '--help'], capture_output=True, text=True)
print('OK' if result.returncode == 0 else 'FAILED')
print(result.stdout[:200] if result.stdout else result.stderr[:200])

In [ ]:
# ============================================================
# Cell 4: クエリファイル生成
# ============================================================

import os, sys
sys.path.insert(0, '/content/toilet-map/batch')

# 生成スクリプトを実行
%cd /content/toilet-map/batch
!python generate_queries.py

# queries.dをGoogle Driveにもコピー
import shutil
src = '/content/toilet-map/batch/queries.d'
dst = f'{BASE}/queries'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# 都道府県一覧表示
prefs = sorted(os.listdir(src))
print(f'\n{len(prefs)}都道府県のクエリ生成完了')
for p in prefs:
    files = os.listdir(os.path.join(src, p))
    print(f'  {p}: {len(files)}バッチ')

print(f'\nGoogle Driveにも保存: {dst}')

In [ ]:
# ============================================================
# Cell 5: スクレイプ実行
# ============================================================

# ---- ここを変更 ----
PREFECTURE = '埼玉県'      # 都道府県名
START_BATCH = 1             # 開始バッチ番号
END_BATCH = 5               # 終了バッチ番号（この番号を含む）
SLEEP_BETWEEN = 60          # クエリ間スリープ（秒）
MAX_RETRIES = 2             # リトライ回数
# -------------------

import subprocess, time, json, os, glob

QUERY_BASE = f'/content/toilet-map/batch/queries.d/{PREFECTURE}'
RAW_DIR = f'{BASE}/raw_parts/{PREFECTURE}'
PROGRESS_DIR = f'{BASE}/progress/{PREFECTURE}'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)

success = 0
failed = 0
skipped = 0

for batch_num in range(START_BATCH, END_BATCH + 1):
    query_file = os.path.join(QUERY_BASE, f'batch_{batch_num:03d}.txt')
    if not os.path.exists(query_file):
        print(f'[batch_{batch_num:03d}] クエリファイルなし - スキップ')
        continue

    # 進捗チェック
    progress_file = os.path.join(PROGRESS_DIR, f'batch_{batch_num:03d}.done')
    output_file = os.path.join(RAW_DIR, f'batch_{batch_num:03d}.json')
    
    if os.path.exists(progress_file) and os.path.exists(output_file):
        print(f'[batch_{batch_num:03d}] (完了済み) スキップ')
        skipped += 1
        continue

    # クエリ読み込み
    with open(query_file, 'r') as f:
        queries = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    
    print(f'\n{"="*50}')
    print(f'[batch_{batch_num:03d}] {len(queries)}クエリ')
    print(f'{"="*50}')
    
    ok = False
    for retry in range(MAX_RETRIES + 1):
        if retry > 0:
            print(f'  リトライ #{retry} ... 120秒待機')
            time.sleep(120)
        
        result = subprocess.run(
            ['google-maps-scraper',
             '-depth', '1',
             '-input', query_file,
             '-results', output_file,
             '-json', '--extra-reviews',
             '-lang', 'ja',
             '-exit-on-inactivity', '5m'],
            capture_output=True, text=True, timeout=600
        )
        
        if result.returncode == 0 and os.path.exists(output_file):
            with open(output_file) as f:
                lines = sum(1 for _ in f)
            if lines > 0:
                print(f'  ✓ 成功 ({lines}件)')
                ok = True
                break
        
        print(f'  ✗ 失敗 (exit={result.returncode})')
        if result.stderr:
            print(f'  stderr: {result.stderr[:200]}')
    
    if ok:
        success += 1
        # 進捗保存
        with open(progress_file, 'w') as f:
            f.write('done')
    else:
        failed += 1
        print(f'  !! 失敗: batch_{batch_num:03d}')
    
    # スリープ
    if batch_num < END_BATCH:
        print(f'  スリープ {SLEEP_BETWEEN}秒 ...')
        time.sleep(SLEEP_BETWEEN)

print(f'\n{"="*50}')
print(f'完了: 成功={success} / 失敗={failed} / スキップ={skipped}')
print(f'{"="*50}')

In [ ]:
# ============================================================
# Cell 6: 処理＆マージ（データをtoilets.jsonに統合）
# ============================================================

import json, glob, os, subprocess, sys

# 全raw_partsをマージ
raw_output = '/content/toilet-map/batch/raw_data.json'
with open(raw_output, 'w') as outf:
    for f in sorted(glob.glob(f'{BASE}/raw_parts/**/*.json', recursive=True)):
        with open(f) as inf:
            outf.write(inf.read())

with open(raw_output) as f:
    total_lines = sum(1 for _ in f)
print(f'raw データ: {total_lines}件')

# process_data.py で差分マージ
processed = '/content/toilet-map/data/toilets.json'
result = subprocess.run(
    [sys.executable, '/content/toilet-map/batch/process_data.py',
     raw_output, processed, '--incremental'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Google Driveにもバックアップ
import shutil
shutil.copy2(processed, f'{BASE}/data/toilets.json')
print(f'\nバックアップ: {BASE}/data/toilets.json')

# 結果確認
with open(processed) as f:
    data = json.load(f)
m = data['metadata']
print(f'\n=== 結果 ===')
print(f'エリア: {m["area_name"]}')
print(f'総数: {m["total"]}件')
print(f'スコアあり: {m["scored"]}件')
print(f'公共トイレ: {m["public_toilets"]}件')

In [ ]:
# ============================================================
# Cell 7: GitHub に push → Streamlit Cloud 自動デプロイ
# ============================================================

import os

# GitHub トークンを入力（初回のみ。Colabのシークレット機能推奨）
from getpass import getpass
TOKEN = getpass('GitHub Personal Access Token (repo権限): ')

# push
%cd /content/toilet-map

!git config user.email 'kaenozu@users.noreply.github.com'
!git config user.name 'kaenozu'

!git add data/toilets.json
!git commit -m 'Update toilet data ({PREFECTURE} batch {START_BATCH}-{END_BATCH})'

# トークンを使ってpush
remote_url = f'https://{TOKEN}@github.com/kaenozu/toilet-map.git'
!git remote set-url origin {remote_url}
!git push origin main

print('\n✅ push完了！ Streamlit Cloudが自動でデプロイします（1〜2分）')